# Quantum autocallable option pricing

Regenerate the three-period grid and apply the strict autocall and barrier rules. Run the full coherent preparation and one local Grover iteration. The long adaptive run remains disabled as in the source. The historical IQAE output is labeled separately from local execution.

## Path-dependent payoff rules stay explicit

The visible native root loads three years of bounded return fields, encodes
the payoff reference and composes the contract's reversible preparation.
FlagQuantum constructs these declared operations and a static Grover family;
classical model.py supplies barriers, strike, discounting, signed encoding
and comparison boundaries for all 64 paths.

Inspect decoded payoff against the independently enumerated expectation and
check both zero and one Grover iterate. The original adaptive estimation run
was disabled; this workflow validates its live preparation path and does not
claim an adaptive price interval or timing from that disabled run. Financial
model discretization remains separate from statevector correctness.


Set the original financial contract and annualize the input statistics.

In [ ]:
from flagquantum.synth import Output
from flagquantum.synth import QBit
from flagquantum.synth import QNum
from flagquantum.synth import allocate
from flagquantum.synth import drop
from flagquantum.synth import phase
from flagquantum.synth import qfunc

from examples.synthesis.applications.finance.autocallable_options.quantum_autocallable_option_pricing.main import (
    prepare,
    reflect,
)


@qfunc
def preparation(
    k: int,
    probabilities: list[float],
    exponential: list[float],
    bounds: list[int],
    indicator: Output[QBit],
) -> None:
    x0: QNum = QNum()
    x1: QNum = QNum()
    x2: QNum = QNum()
    aux: QNum = QNum()
    allocate(2, x0)
    allocate(2, x1)
    allocate(2, x2)
    allocate(4, aux)
    allocate(indicator)
    prepare(
        probabilities,
        exponential,
        bounds[0],
        bounds[1],
        bounds[2],
        bounds[3],
        bounds[4],
        bounds[5],
        bounds[6],
        x0,
        x1,
        x2,
        aux,
        indicator,
    )
    for iteration in range(k):
        phase(indicator, 3.141592653589793)
        reflect(
            probabilities,
            exponential,
            bounds[0],
            bounds[1],
            bounds[2],
            bounds[3],
            bounds[4],
            bounds[5],
            bounds[6],
            x0,
            x1,
            x2,
            aux,
            indicator,
        )
    drop(x0)
    drop(x1)
    drop(x2)
    drop(aux)


import math

import examples.synthesis.applications.finance.autocallable_options.quantum_autocallable_option_pricing.overview.model as model
import examples.synthesis.applications.finance.autocallable_options.quantum_autocallable_option_pricing.overview.workflow as workflow

print(model.NOTIONAL, model.YEARS, model.RATE)
print(model.ANNUAL_DRIFT, model.ANNUAL_VOLATILITY)


Use the original two fractional bits.

In [ ]:
assert model.FRACTION_BITS == 2


Integrate the Gaussian mass of each two-bit bin.

In [ ]:
grid = model.gaussian_grid()
print(grid.points, grid.probabilities)


Bound the log return and the call payoff normalization.

In [ ]:
transform = model.encoding(grid)
print(transform.return_bound)


Set the fixed-point step.

In [ ]:
step = 1 / model.UNIT
print(step)


Size the signed integer part.

In [ ]:
print(transform.integer_bits, transform.width)


Derive the exponential normalization factors.

In [ ]:
print(transform.numerator, transform.denominator)


Calculate the financial scale of the probability encoding.

In [ ]:
print(transform.normalization)


Generate the three constant marker rotations.

In [ ]:
rotations = tuple(transform.rotation(payoff) for payoff in (*model.CALL_PAYMENTS, 0.0))
print(rotations)


Check that all constant rotations are in range.

In [ ]:
assert all(0 <= rotation <= math.pi for rotation in rotations)


Decode an amplitude probability into a discounted payoff.

In [ ]:
decode = transform.decode


Inspect the first discounted call payment.

In [ ]:
print(model.CALL_PAYMENTS[0])


Round-trip the first call payment through its rotation.

In [ ]:
assert math.isclose(
    decode(math.sin(rotations[0] / 2) ** 2), model.CALL_PAYMENTS[0], abs_tol=1e-12
)


Inspect the second discounted call payment.

In [ ]:
print(model.CALL_PAYMENTS[1])


Round-trip the second call payment.

In [ ]:
assert math.isclose(
    decode(math.sin(rotations[1] / 2) ** 2), model.CALL_PAYMENTS[1], abs_tol=1e-12
)


Round-trip the zero payoff.

In [ ]:
assert math.isclose(decode(math.sin(rotations[2] / 2) ** 2), 0, abs_tol=1e-12)


Take the original log barriers and strike.

In [ ]:
print(math.log(model.BARRIER), math.log(model.STRIKE / model.NOTIONAL))


Enumerate all paths through the four bins over three years.

In [ ]:
paths = model.paths(grid)
assert len(paths) == 64


Express the fixed-point resolution in raw units.

In [ ]:
assert model.UNIT * step == 1


Match source coefficient truncation toward zero.

In [ ]:
assert (grid.raw_offset, grid.raw_slope) == (-2, 1)


Accumulate the returns for the three periods.

In [ ]:
print("Extreme raw log returns:", paths[0].log_returns, paths[-1].log_returns)


Check the mass of the joint path distribution.

In [ ]:
assert math.isclose(math.fsum(row.probability for row in paths), 1.0, abs_tol=1e-12)


Check the payoff priority and strict barriers.

In [ ]:
assert {row.payment for row in paths} == set(model.Payment)


Weight each payoff by its path probability.

In [ ]:
expectation = math.fsum(row.probability * row.discounted_payoff for row in paths)


Report the classical finite-grid price.

In [ ]:
print("Classical enumerated payoff:", expectation)


Generate the normalized exponential reference distribution.

In [ ]:
reference_weights = transform.reference_probabilities()
assert len(reference_weights) == 16
assert math.isclose(math.fsum(reference_weights), 1.0)


Check the four coefficient-quantized increments.

In [ ]:
print(tuple((grid.raw_offset + grid.raw_slope * word) * step for word in range(4)))


Translate the signed terminal return into the unsigned reference register.

In [ ]:
assert transform.width == 4
sign_bias = 1 << (transform.width - 1)
print(sign_bias)


Use strict unsigned equivalents of the signed source comparisons.

In [ ]:
print(model.comparisons(grid, transform))


Bind fresh distributions to the native coherent payoff and reflection.

In [ ]:
builder = workflow.build_program


Account for all persistent registers of the native preparation.

In [ ]:
logical_width = model.YEARS * model.GRID_BITS + transform.width + 1
assert logical_width == 11


Synthesize and execute the preparation and one static Grover iterate locally.

In [ ]:
result = workflow.run()
assert result.expectation == expectation


Report width and check every physical amplitude against the independent construction.

In [ ]:
assert all(
    check.state_error < 1e-12 and check.workspace_probability < 1e-24
    for check in result.checks
)
print("Physical widths:", tuple(check.resources.n_qubits for check in result.checks))


Keep the source's long adaptive IQAE experiment disabled.

In [ ]:
# Original disabled run configuration: epsilon = 0.001, alpha = 0.002.


Keep the disabled adaptive estimator's output and timings separate from the executed local result.

In [ ]:
# The original adaptive payoff, interval, and timings were conditional on that disabled run.


Report the calculated price, executed quantum checks and source-reported historical comparison.

In [ ]:
workflow.print_report(result)
